##Configure the Gold layer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

CATALOG = "capretailnew"

SILVER = CATALOG + ".silver"
GOLD = CATALOG + ".gold"

# Create Gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS " + GOLD)

print("Gold schema created successfully")

##Load all six Silver tables

In [0]:
sales = spark.table(
    SILVER + ".silver_sales_order_lines"
)

inventory = spark.table(
    SILVER + ".silver_inventory_movements"
)

returns = spark.table(
    SILVER + ".silver_sales_returns"
)

products = spark.table(
    SILVER + ".silver_product_master"
)

stores = spark.table(
    SILVER + ".silver_store_master"
)

suppliers = spark.table(
    SILVER + ".silver_supplier_master"
)

print("All Silver tables loaded successfully")

##Verify the input data

In [0]:
print("Sales:", sales.count())
print("Inventory:", inventory.count())
print("Returns:", returns.count())

print("Products:", products.count())
print("Stores:", stores.count())
print("Suppliers:", suppliers.count())

##Inspect the columns

In [0]:
sales.printSchema()
inventory.printSchema()
returns.printSchema()
products.printSchema()
stores.printSchema()
suppliers.printSchema()

##Create Product Dimension

In [0]:
# Select required product columns
product_data = products.select(
    "product_id",
    "sku",
    "product_name",
    "category",
    "subcategory",
    "unit_of_measure",
    "list_price",
    "cost_price",
    "supplier_id",
    "active_flag"
)

# Select supplier details
supplier_data = suppliers.select(
    "supplier_id",
    "supplier_name",
    "lead_time_days",
    "supplier_tier"
)

# Join product and supplier
gold_dim_product = product_data.join(
    supplier_data,
    on="supplier_id",
    how="left"
)

# Save Gold table
gold_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_dim_product")

print("Product dimension created")

##Create Store Dimension

In [0]:
gold_dim_store = stores.select(
    "store_id",
    "store_name",
    "city",
    "region",
    "store_type",
    "opening_date",
    "active_flag",
    "warehouse_id",
    "manager_code",
    "country"
)

gold_dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_dim_store")

print("Store dimension created")

##Create Supplier Dimension

In [0]:
gold_dim_supplier = suppliers.select(
    "supplier_id",
    "supplier_name",
    "country",
    "lead_time_days",
    "payment_terms",
    "active_flag",
    "supplier_tier",
    "city",
    "category",
    "contact_code"
)

gold_dim_supplier.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD + ".gold_dim_supplier")

print("Supplier dimension created")

##Validate dimension tables

In [0]:
display(
    spark.table(GOLD + ".gold_dim_product")
)

display(
    spark.table(GOLD + ".gold_dim_store")
)

display(
    spark.table(GOLD + ".gold_dim_supplier")
)

In [0]:
print(
    "Gold Products:",
    spark.table(GOLD + ".gold_dim_product").count()
)

print(
    "Gold Stores:",
    spark.table(GOLD + ".gold_dim_store").count()
)

print(
    "Gold Suppliers:",
    spark.table(GOLD + ".gold_dim_supplier").count()
)